### Star Schema

In [1]:
%%sql

CREATE SCHEMA IF NOT EXISTS gold;


CREATE TABLE IF NOT EXISTS gold.Dim_Identity (
    Identity_Key INT, 
    First_Name STRING,
    Last_Name STRING,
    Email STRING,
    Gender STRING,
    Age INT
) USING DELTA;


CREATE TABLE IF NOT EXISTS gold.Dim_Academic_Context (
    Academic_Key INT,
    Department STRING,
    Grade STRING,
    Extracurricular_Activities STRING
) USING DELTA;

CREATE TABLE IF NOT EXISTS gold.Dim_Social (
    Social_Key INT,
    Internet_Access_at_Home STRING,
    Parent_Education_Level STRING,
    Family_Income_Level STRING
) USING DELTA;


CREATE TABLE IF NOT EXISTS gold.Dim_Assignments (
    Assignment_Key INT,
    assi_Math STRING,
    assi_Science STRING,
    assi_English STRING,
    assi_Social_Studie STRING,
    assi_French STRING,
    assi_Computer_Science STRING,
    assi_late STRING
) USING DELTA;


CREATE TABLE IF NOT EXISTS gold.Fact_Performance (
    Fact_Key INT,
    Identity_Key INT,
    Academic_Key INT,
    Social_Key INT,
    Assignment_Key INT,
    Student_ID STRING,
    Math_Score DECIMAL(10, 2),
    Science_Score DECIMAL(10, 2),
    English_Score DECIMAL(10, 2),
    Social_Studies_Score DECIMAL(10, 2),
    French_Score DECIMAL(10, 2),
    Computer_Science_Score DECIMAL(10, 2),
    last_score DECIMAL(10, 2),
    Total_Avg DECIMAL(10, 2),
    Study_Hours_per_Week DOUBLE,
    Stress_Level_1_10 INT,
    Sleep_Hours_per_Night DOUBLE,
    academic_attendance DOUBLE,
    non_academic_attendance DOUBLE,
    lms_login_freq_per_day DOUBLE,
    lms_active_avg_hrs DOUBLE,
    resource_access_avg_hrs DOUBLE
) USING DELTA;

StatementMeta(, 36a0b9ed-87e1-4112-8cad-30a9bbd57653, 7, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

### Data Ingestion

In [2]:
%%sql
INSERT INTO gold.Dim_Identity 
SELECT 
    id AS Identity_Key, 
    First_Name, 
    Last_Name, 
    Email, 
    Gender, 
    Age 
FROM silver.silver_students;

INSERT INTO gold.Dim_Academic_Context 
SELECT 
    id AS Academic_Key, 
    Department, 
    Grade, 
    Extracurricular_Activities 
FROM silver.silver_students;

INSERT INTO gold.Dim_Social 
SELECT 
    id AS Social_Key, 
    Internet_Access_at_Home, 
    Parent_Education_Level, 
    Family_Income_Level 
FROM silver.silver_students;

INSERT INTO gold.Dim_Assignments 
SELECT 
    id AS Assignment_Key, 
    assi_Math, 
    assi_Science, 
    assi_English, 
    assi_Social_Studie, 
    assi_French, 
    assi_Computer_Science, 
    assi_late 
FROM silver.silver_students;

INSERT INTO gold.Fact_Performance 
SELECT 
    id AS Fact_Key,
    id AS Identity_Key,    
    id AS Academic_Key,    
    id AS Social_Key,     
    id AS Assignment_Key, 
    Student_ID, 
    Math_Score, 
    Science_Score, 
    English_Score, 
    Social_Studies_Score, 
    French_Score, 
    Computer_Science_Score, 
    last_score, 
    Total_Avg, 
    Study_Hours_per_Week, 
    Stress_Level_1_10,      
    Sleep_Hours_per_Night, 
    academic_attendance, 
    non_academic_attendance, 
    lms_login_freq_per_day, 
    lms_active_avg_hrs, 
    resource_access_avg_hrs 
FROM silver.silver_students;%%sql
INSERT INTO gold.Dim_Identity 
SELECT 
    id AS Identity_Key, 
    First_Name, 
    Last_Name, 
    Email, 
    Gender, 
    Age 
FROM silver.silver_students;

INSERT INTO gold.Dim_Academic_Context 
SELECT 
    id AS Academic_Key, 
    Department, 
    Grade, 
    Extracurricular_Activities 
FROM silver.silver_students;

INSERT INTO gold.Dim_Social 
SELECT 
    id AS Social_Key, 
    Internet_Access_at_Home, 
    Parent_Education_Level, 
    Family_Income_Level 
FROM silver.silver_students;

INSERT INTO gold.Dim_Assignments 
SELECT 
    id AS Assignment_Key, 
    assi_Math, 
    assi_Science, 
    assi_English, 
    assi_Social_Studie, 
    assi_French, 
    assi_Computer_Science, 
    assi_late 
FROM silver.silver_students;

INSERT INTO gold.Fact_Performance 
SELECT 
    id AS Fact_Key,
    id AS Identity_Key,    
    id AS Academic_Key,    
    id AS Social_Key,     
    id AS Assignment_Key, 
    Student_ID, 
    Math_Score, 
    Science_Score, 
    English_Score, 
    Social_Studies_Score, 
    French_Score, 
    Computer_Science_Score, 
    last_score, 
    Total_Avg, 
    Study_Hours_per_Week, 
    Stress_Level_1_10,      
    Sleep_Hours_per_Night, 
    academic_attendance, 
    non_academic_attendance, 
    lms_login_freq_per_day, 
    lms_active_avg_hrs, 
    resource_access_avg_hrs 
FROM silver.silver_students;

StatementMeta(, 36a0b9ed-87e1-4112-8cad-30a9bbd57653, 12, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

### Aggregated Data Marts

In [3]:
%%sql

DROP TABLE IF EXISTS gold.academic_demographics_analysis;
CREATE TABLE gold.academic_demographics_analysis USING DELTA AS
SELECT 
    a.Department, a.Grade, i.Gender, i.Age,
    f.Math_Score, f.Science_Score, f.English_Score, f.Social_Studies_Score, 
    f.French_Score, f.Computer_Science_Score, f.Total_Avg
FROM gold.Fact_Performance f
JOIN gold.Dim_Identity i ON f.Identity_Key = i.Identity_Key
JOIN gold.Dim_Academic_Context a ON f.Academic_Key = a.Academic_Key;

DROP TABLE IF EXISTS gold.socioeconomic_impact;
CREATE TABLE gold.socioeconomic_impact USING DELTA AS
SELECT 
    s.Parent_Education_Level, s.Family_Income_Level, s.Internet_Access_at_Home,
    a.Grade, f.lms_active_avg_hrs, f.resource_access_avg_hrs
FROM gold.Dim_Social s
JOIN gold.Fact_Performance f ON s.Social_Key = f.Social_Key
JOIN gold.Dim_Academic_Context a ON f.Academic_Key = a.Academic_Key;

DROP TABLE IF EXISTS gold.study_habits_stress_analysis;
CREATE TABLE gold.study_habits_stress_analysis USING DELTA AS
SELECT 
    f.Study_Hours_per_Week, f.Sleep_Hours_per_Night, f.Stress_Level_1_10,
    a.Grade, a.Extracurricular_Activities
FROM gold.Fact_Performance f
JOIN gold.Dim_Academic_Context a ON f.Academic_Key = a.Academic_Key;

DROP TABLE IF EXISTS gold.lms_engagement_analysis;
CREATE TABLE gold.lms_engagement_analysis USING DELTA AS
SELECT 
    f.lms_login_freq_per_day, f.lms_active_avg_hrs, f.resource_access_avg_hrs,
    f.Math_Score, f.Science_Score, a.Grade
FROM gold.Fact_Performance f
JOIN gold.Dim_Academic_Context a ON f.Academic_Key = a.Academic_Key;

DROP TABLE IF EXISTS gold.attendance_and_submission_behavior;
CREATE TABLE gold.attendance_and_submission_behavior USING DELTA AS
SELECT 
    f.academic_attendance, f.non_academic_attendance, f.Stress_Level_1_10, f.lms_login_freq_per_day,
    a.Grade, a.Extracurricular_Activities,
    d.assi_late
FROM gold.Fact_Performance f
JOIN gold.Dim_Academic_Context a ON f.Academic_Key = a.Academic_Key
JOIN gold.Dim_Assignments d ON f.Assignment_Key = d.Assignment_Key;

DROP TABLE IF EXISTS gold.subject_performance_comparison;
CREATE TABLE gold.subject_performance_comparison USING DELTA AS
SELECT 
    f.Math_Score, f.Science_Score, f.Computer_Science_Score, f.English_Score, f.French_Score,
    d.assi_Math, d.assi_Science, d.assi_English, d.assi_Social_Studie, d.assi_French, d.assi_Computer_Science
FROM gold.Fact_Performance f
JOIN gold.Dim_Assignments d ON f.Assignment_Key = d.Assignment_Key;

StatementMeta(, 36a0b9ed-87e1-4112-8cad-30a9bbd57653, 24, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

...

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>